In [ ]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index
from dotenv import load_dotenv
from groq import Groq

try:
    load_dotenv()
except Exception as e:
    print(f"Error loading .env file: {e}")

groq_client = Groq()


messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = groq_client.chat.completions.create(
    model="groq/compound-mini",
    messages=messages
)

response.choices[0].message.content


In [ ]:
from ingest import load_faq_data, build_index
documents = load_faq_data()
index = build_index(documents)

index.search("I just discovered the course. Can I join it?")

In [3]:
def search(query):
    boost_dict = {"question": 3.0, "section": 1}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [ ]:
search("Can I use Ollama models?")

In [ ]:
index.search(
    "Can I use Ollama models?",
    filter_dict={"course": "llm-zoomcamp"},  # ← aggiungi il filtro
    num_results=5
)

In [ ]:
# Formato OpenAI Define the tool specification for the search function ()
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [ ]:
# Formato Groq Define the tool specification for the search function ()
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"]
            # ← rimosso additionalProperties: False
        }
    }
}

In [ ]:
from groq import Groq
from dotenv import load_dotenv

try:
    load_dotenv()
except Exception as e:
    print(f"Error loading .env file: {e}")

groq_client = Groq()

question = "I just discovered the course. Can I still join?"
messages = [
    {"role": "user", "content": question}
]

response = groq_client.chat.completions.create(
    model="llama-3.1-8b-instant",  # ← modello Groq
    messages=messages,
    tools=[search_tool]
)

response.choices[0].message.content

In [ ]:
response

In [ ]:
import json

# Step 1: già fatto — ottieni la tool call
tool_call = response.choices[0].message.tool_calls[0]
print(f'Tool call: {tool_call}')
query = json.loads(tool_call.function.arguments)["query"]
print(f'Query: {query}')

# Step 2: esegui il tool
search_results = search(query)
print(f'Search results: {search_results}')

# Step 3: aggiungi la risposta del tool alla history
messages.append(response.choices[0].message)  # risposta assistant
print(f'Messages after assistant response: {messages}')
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(search_results)      # risultati della ricerca
})
print(f'Messages after tool response: {messages}')

# Step 4: seconda chiamata al LLM con i risultati
final_response = groq_client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    tools=[search_tool]
)

print(final_response.choices[0].message.content)

In [ ]:
print(f"finish_reason: {final_response.choices[0].finish_reason}")
print(f"content: {final_response.choices[0].message.content}")
print(f"tool_calls: {final_response.choices[0].message.tool_calls}")

In [ ]:
usage = final_response.usage
print(usage)
print(f"completion_tokens: {usage.completion_tokens}")
print(f"prompt_tokens: {usage.prompt_tokens}")
print(f"total_tokens: {usage.total_tokens}")

In [ ]:
def calculate_groq_cost(prompt_tokens: int, completion_tokens: int) -> dict:
    """
    Calcola il costo di una chiamata API Groq con llama-3.1-8b-instant.
    
    Prezzi (giugno 2026):
      - Input:  $0.05 per 1M token
      - Output: $0.08 per 1M token
    """
    INPUT_PRICE_PER_M  = 0.05  # $ per milione di token
    OUTPUT_PRICE_PER_M = 0.08  # $ per milione di token

    input_cost  = (prompt_tokens     / 1_000_000) * INPUT_PRICE_PER_M
    output_cost = (completion_tokens / 1_000_000) * OUTPUT_PRICE_PER_M
    total_cost  = input_cost + output_cost

    return {
        "prompt_tokens":     prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens":      prompt_tokens + completion_tokens,
        "input_cost":        round(input_cost,  8),
        "output_cost":       round(output_cost, 8),
        "total_cost":        round(total_cost,  8),
    }




In [ ]:
# Utilizzo con la response di Groq
usage = final_response.usage
cost = calculate_groq_cost(
    prompt_tokens=usage.prompt_tokens,
    completion_tokens=usage.completion_tokens
)

print(f"Prompt tokens:     {cost['prompt_tokens']}")
print(f"Completion tokens: {cost['completion_tokens']}")
print(f"Total tokens:      {cost['total_tokens']}")
print(f"Input cost:        ${cost['input_cost']:.8f}")
print(f"Output cost:       ${cost['output_cost']:.8f}")
print(f"Total cost:        ${cost['total_cost']:.8f}")

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()



In [ ]:
instructions = """
You're a course teaching assistant.
Answer student questions using the search function to look up information.
Always use the 'search' tool before answering.
""".strip()

In [ ]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [ ]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False
    response = groq_client.chat.completions.create(
        #model="llama-3.1-8b-instant",
        model="llama-3.3-70b-versatile",
        #model = "openai/gpt-oss-20b",
        messages=messages,
        tools=[search_tool],
        tool_choice={
        "type": "function",
        "function": {"name": "search"}  # ← nome esatto del tuo tool
        }
    )

    choice = response.choices[0]
    messages.append(choice.message)         # aggiungi il messaggio assistant
    has_function_calls = False

    for tool_call in choice.message.tool_calls:
        if choice.finish_reason == "tool_calls":    # tool call
            print("function_call:", tool_call.function.name, tool_call.function.arguments)
            for tool_call in choice.message.tool_calls:
                search_results = search(
                    json.loads(tool_call.function.arguments)["query"]
                )
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(search_results)
                })
                has_function_calls = True

        elif choice.finish_reason == "stop":        # risposta testuale
            print("assistant response:", choice.message.content)
            #print(choice.message.content)
    it = it + 1
    if has_function_calls == False:
        break        


In [ ]:
response

In [ ]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index
from dotenv import load_dotenv
from groq import Groq

try:
    load_dotenv()
except Exception as e:
    print(f"Error loading .env file: {e}")

groq_client = Groq()

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

it = 1

while True:
    print(f"iteration #{it}...")

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        tools=[search_tool],
        # ← rimosso tool_choice forzato: il modello decide quando fermarsi
    )

    choice = response.choices[0]
    messages.append(choice.message)

    if choice.finish_reason == "tool_calls":
        for tool_call in choice.message.tool_calls:
            print("function_call:", tool_call.function.name, tool_call.function.arguments)
            search_results = search(
                json.loads(tool_call.function.arguments)["query"]
            )
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(search_results)
            })

    elif choice.finish_reason == "stop":
        print("assistant response:", choice.message.content)
        break  # ← esce dal loop quando il modello ha risposto

    it += 1

In [ ]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index
from dotenv import load_dotenv
from groq import Groq

try:
    load_dotenv()
except Exception as e:
    print(f"Error loading .env file: {e}")

groq_client = Groq()

In [5]:
# Formato Groq Define the tool specification for the search function ()
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"]
            # ← rimosso additionalProperties: False
        }
    }
}

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()



In [ ]:
import json
import time

def agent_loop(instructions, question, model="llama-3.3-70b-versatile", max_retries=3):
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    it = 1
    max_retries = 3

    while True:
        print(f"iteration #{it}...")

        # retry in caso di tool_use_failed
        for attempt in range(max_retries):
            try:
                response = groq_client.chat.completions.create(
                    model="llama-3.3-70b-versatile",
                    messages=messages,
                    tools=[search_tool],
                    temperature=0.0,  # ← più deterministico, riduce il bug
                    extra_body={"disable_tool_validation": True}  # ← bypassa la validazione Groq
                )
                break  # successo, esci dal retry loop
            except Exception as e:
                if "tool_use_failed" in str(e) and attempt < max_retries - 1:
                    print(f"  ⚠️ tool_use_failed, retry {attempt + 1}/{max_retries}...")
                    time.sleep(0.5)
                else:
                    raise  # rilancia se non è tool_use_failed o esauriti i retry

        choice = response.choices[0]
        messages.append(choice.message)

        if choice.finish_reason == "tool_calls":
            for tool_call in choice.message.tool_calls:
                print("function_call:", tool_call.function.name, tool_call.function.arguments)
                search_results = search(
                    json.loads(tool_call.function.arguments)["query"]
                )
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(search_results)
                })

        elif choice.finish_reason == "stop":
            print("assistant response:", choice.message.content)
            break

        it += 1

In [ ]:
agent_loop(instructions, "How do I run Ollama locally?")

In [ ]:
agent_loop(instructions, "Can I use Gemini LLM?")

In [109]:
import json
import re
import time

def parse_failed_generation(error_str):
    """
    Gestisce due casi di failed_generation:
    1. Tag malformato:  <function=search ...> → esegui il tool
    2. Testo puro:      "You can join..." → restituisci come risposta finale
    """
    # Caso 1: tag function malformato
    match = re.search(
        r"'failed_generation': '<function=(\w+)[= ]*(\{.*?\})\s*</function>'",
        error_str
    )
    if match:
        return "tool_call", match.group(1), match.group(2)

    # Caso 2: risposta testuale diretta
    match = re.search(
        r"'failed_generation': ['\"](.+?)['\"]}}",
        error_str,
        re.DOTALL
    )
    if match:
        return "text_response", None, match.group(1)

    return None, None, None

def agent_loop(instructions, question, model="llama-3.3-70b-versatile", max_iterations=10):
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    for it in range(1, max_iterations + 1):
        print(f"iteration #{it}...")

        try:
            response = groq_client.chat.completions.create(
                model=model,
                messages=messages,
                tools=[search_tool],
                tool_choice="required"
            )
            choice = response.choices[0]

        except Exception as e:
            error_str = str(e)
            if "tool_use_failed" not in error_str:
                raise

            print("  ⚠️ tool_use_failed: parsing failed_generation manually...")
            case, name, content = parse_failed_generation(error_str)  # ← ora ritorna 3 valori

            if case == "tool_call":
                # Caso 1: tag malformato → esegui il tool
                print(f"  → Recovered tool call: {name}({content})")
                query = json.loads(content)["query"]
                search_results = search(query)
                messages.append({
                    "role": "assistant",
                    "content": None,
                    "tool_calls": [{
                        "id": f"recovered_{it}",
                        "type": "function",
                        "function": {"name": name, "arguments": content}
                    }]
                })
                messages.append({
                    "role": "tool",
                    "tool_call_id": f"recovered_{it}",
                    "content": json.dumps(search_results)
                })
                continue

            elif case == "text_response":
                # Caso 2: il modello ha risposto con testo → accettalo come risposta finale
                print(f"  → Recovered text response")
                print(f"assistant response: {content}")
                return content

            else:
                raise ValueError(f"Cannot parse failed_generation from: {error_str}")

        # Caso normale
        messages.append(choice.message)

        if choice.finish_reason == "tool_calls":
            for tool_call in choice.message.tool_calls:
                print(f"  function_call: {tool_call.function.name} {tool_call.function.arguments}")
                search_results = search(json.loads(tool_call.function.arguments)["query"])
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(search_results)
                })

        elif choice.finish_reason == "stop":
            print("assistant response:", choice.message.content)
            return choice.message.content

    print("⚠️ Raggiunto il limite di iterazioni")

In [ ]:
agent_loop(instructions, "How do I run Ollama locally?")

In [ ]:
agent_loop(instructions, "Can I use Gemini LLM?")

In [ ]:
agent_loop(instructions, "Is there any alternative to ChatGPT LLM?")

In [ ]:
agent_loop(instructions, "I discovered llm course late. Can I join it?")

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

In [ ]:
agent_loop(instructions, "what's queen gambit?")

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

NEVER answer without searching first!

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

In [110]:
agent_loop(instructions, "I just discovered the course. Can I join it?", model="qwen/qwen3-32b")

iteration #1...
  function_call: search {"query":"Can I join the course late?"}
iteration #2...
  ⚠️ tool_use_failed: parsing failed_generation manually...
  → Recovered text response
assistant response: You can join the course even if you just discovered it! To clarify:\n\n1. **Joining Late**: There\'s no strict deadline to join the course itself. You\'re welcome to start learning and completing assignments at any time.\n\n2. **Certificate Eligibility**:  \n   To earn a certificate, you must:\n   - Submit your capstone project **while the submission form is open** (typically during the live cohort period). \n   - Participate in **peer reviews** (3 required) **during the course runtime**, as these are only enabled while the course is active.  \n   - Note: Certificates are only issued for live cohorts, not self-paced enrollment.\n\n3. **Next Course Offering**: If you miss this cohort, the course will be offered again in **Summer 2025**.\n\nYou don’t need a separate confirmation email to

'You can join the course even if you just discovered it! To clarify:\\n\\n1. **Joining Late**: There\\\'s no strict deadline to join the course itself. You\\\'re welcome to start learning and completing assignments at any time.\\n\\n2. **Certificate Eligibility**:  \\n   To earn a certificate, you must:\\n   - Submit your capstone project **while the submission form is open** (typically during the live cohort period). \\n   - Participate in **peer reviews** (3 required) **during the course runtime**, as these are only enabled while the course is active.  \\n   - Note: Certificates are only issued for live cohorts, not self-paced enrollment.\\n\\n3. **Next Course Offering**: If you miss this cohort, the course will be offered again in **Summer 2025**.\\n\\nYou don’t need a separate confirmation email to begin—just start learning and submit work while the submission window is open. Registration is optional for "tracking purposes" but not required.'

TEST UTILIZZO di API ANTHROPIC

In [111]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

# ── Client Anthropic ──────────────────────────────────────────
anthropic_client = anthropic.Anthropic()

# ── Definizione tool (stessa logica, formato leggermente diverso) ──
# Anthropic usa "input_schema" invece di "parameters"
search_tool_anthropic = {
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "input_schema": {                      # ← "input_schema" invece di "parameters"
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"]
    }
}

# ── Agent loop ────────────────────────────────────────────────
def agent_loop(instructions, question, model="claude-haiku-4-5", max_iterations=10):
    messages = [
        {"role": "user", "content": question}
    ]

    for it in range(1, max_iterations + 1):
        print(f"iteration #{it}...")

        response = anthropic_client.messages.create(
            model=model,
            max_tokens=1024,
            system=instructions,           # ← system prompt separato dai messages
            tools=[search_tool_anthropic],
            messages=messages
        )

        if response.stop_reason == "tool_use":
            # Aggiungi la risposta dell'assistant alla history
            messages.append({
                "role": "assistant",
                "content": response.content  # ← lista di blocchi (testo + tool_use)
            })

            # Processa tutti i tool call nella risposta
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  function_call: {block.name} {json.dumps(block.input)}")
                    search_results = search(block.input["query"])
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,           # ← "tool_use_id" invece di "tool_call_id"
                        "content": json.dumps(search_results)
                    })

            # Aggiungi i risultati come messaggio "user" (diverso da Groq!)
            messages.append({
                "role": "user",            # ← Anthropic usa "user" per i tool results
                "content": tool_results
            })

        elif response.stop_reason == "end_turn":
            # Estrai il testo dalla risposta
            answer = next(
                (block.text for block in response.content if block.type == "text"),
                None
            )
            print(f"assistant response: {answer}")
            return answer

    print("⚠️ Raggiunto il limite di iterazioni")

In [112]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

NEVER answer without searching first!

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
  function_call: search {"query": "queen gambit"}
iteration #2...
assistant response: The search didn't return any results related to "queen gambit" in the course FAQ database. This appears to be an off-topic question, as "Queen's Gambit" is a chess opening or a TV series, and doesn't relate to the course material or its logistics.

If you have questions about the course content, assignments, grading, deadlines, or other course-related matters, I'd be happy to help! Is there anything course-related I can assist you with?


'The search didn\'t return any results related to "queen gambit" in the course FAQ database. This appears to be an off-topic question, as "Queen\'s Gambit" is a chess opening or a TV series, and doesn\'t relate to the course material or its logistics.\n\nIf you have questions about the course content, assignments, grading, deadlines, or other course-related matters, I\'d be happy to help! Is there anything course-related I can assist you with?'

In [1]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [ ]:
def search(query):
    boost_dict = {"question": 3.0, "section": 1}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [ ]:
# Formato Groq Define the tool specification for the search function ()
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"]
            # ← rimosso additionalProperties: False
        }
    }
}

In [6]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [9]:
search_tool

{'type': 'function',
 'function': {'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course FAQ.'}},
   'required': ['query']}}}

In [10]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [11]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [12]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [23]:
from minsearch import Index
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)
index.fit(documents)

In [24]:
from anthropic_kit import AnthropicClient, Tools, IPythonChatInterface
from anthropic_kit import AnthropicRunner, DisplayingRunnerCallback

# Auto-generazione schema da type hints + docstring
def search(query: str) -> list:
    """Search the FAQ database for entries matching the given query."""
    return index.search(query, num_results=5,
                        boost_dict={"question": 3.0, "section": 0.5},
                        filter_dict={"course": "llm-zoomcamp"})

agent_tools = Tools()
agent_tools.add_tool(search)

# Puoi vedere lo schema generato automaticamente
agent_tools.get_tools()

[{'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'input_schema': {'type': 'object',
   'properties': {'query': {'type': 'string', 'description': 'query'}},
   'required': ['query']}}]

In [29]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

# Definisci le istruzioni QUI, prima di creare il runner
instructions = """
You're a course teaching assistant.
Answer student questions using the search function to look up information.
Search once, and only search again if the first result is not relevant.
Only use facts from the FAQ database.
""".strip()

runner = AnthropicRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=AnthropicClient(model="claude-haiku-4-5"),
)

# Singolo prompt
result = runner.loop(prompt="Can I use Groq instead of ChatGPT?", callback=callback)
print(result.cost)
print(result.tokens)

→ Response received

**Assistant:** I'll search the FAQ database for information about using Groq as an alternative to ChatGPT.

🔍 **Tool call:** `search`
```json
{
  "query": "Groq ChatGPT alternative"
}
```
**Result:** [{"id": "ee43413718", "course": "llm-zoomcamp", "section": "Module 1: Introduction to LLMs and RAG", "question": "OpenSource: Can I use Groq instead of OpenAI?", "answer": "You can use any LLM platform for your experiments and your project. The homework is designed so that you don\u2019t need access...

→ Response received

**Assistant:** Yes, you can use Groq instead of ChatGPT/OpenAI! According to the FAQ:

**You can use any LLM platform for your experiments and your project.** The homework is designed so that you don't need access to any paid services and can complete it locally.

However, there are a couple of things to keep in mind:

1. **You'll need to adjust the code** - Since Groq has different APIs and interfaces than OpenAI, you'll need to modify the code to work with Groq's platform. Refer to Groq's documentation pages for guidance on how to do this.

2. **For token counting** - If you need to count tokens (like in homework assignments), note that Groq doesn't provide a tokenizer library. In that case, you can use the `tiktoken` Python library, which is OpenAI's tokenizer. You don't need an OpenAI API key to use it—you can use it just to estimate token counts.

So while you can definitely use Groq, just be prepared to adapt the provided code to work with their API.

CostInfo(model='claude-haiku-4-5', input_tokens=1624, output_tokens=328, total_cost=$0.002611)
TokenUsage(model='claude-haiku-4-5', input_tokens=1624, output_tokens=328)
